# Driving: real mini data to a controlled comparison

This notebook contains the project source. It can run in Jupyter or a
suitable Colab runtime without a GitHub upload or source-repository login.
Its steps are acquisition, CPU review, one pinned GPU extraction, then
four CPU models and the existing explorer.

**Verification boundary:** source bootstrap, archive handling, review
persistence and CPU comparisons have fixture checks. This notebook has
not completed a real nuScenes download or Cosmos GPU run in the managed
workspace. No real score or GPU fit is claimed.

Use an environment whose own network policy permits the official data
and model hosts. The cells preserve inherited proxies and TLS checks;
a blocked download fails normally. No account, paid runtime or resource
booking is created. Review data/model terms at their official sources.
Mini has ten scenes and no official test split: this workflow's frozen
6/2/2 allocation is a custom proof of concept.

Cells that acquire data or download a model do real work when executed.
Keep WORK on storage you retain across runtime restarts. Review decisions
save there immediately. Preserve the whole WORK directory for transfer.


In [ ]:
import base64, hashlib, io, json, os, shlex, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath

SOURCE_PAYLOAD = '...'
SOURCE_SHA256 = '56a3236dcec0b0b806abe79368daa0d3dd46927b1a7499f331a9eb0fb8a30c14'
WORK = Path(os.environ.get("DRIVING_WORK_DIR",
    "/content/driving-work" if Path("/content").is_dir() else "driving-work")).resolve()
ROOT = WORK / "source"
WORK.mkdir(parents=True, exist_ok=True)
raw_source = base64.b64decode(SOURCE_PAYLOAD)
assert hashlib.sha256(raw_source).hexdigest() == SOURCE_SHA256
with zipfile.ZipFile(io.BytesIO(raw_source)) as bundle:
    source_manifest = json.loads(bundle.read("SOURCE_MANIFEST.json"))
    for name, expected in source_manifest.items():
        relative = PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe source path")
        data = bundle.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError("Embedded source hash mismatch: " + name)
        path = ROOT / name
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError("Existing source differs; preserve it and choose a fresh WORK")
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            path.write_bytes(data)
sys.path.insert(0, str(ROOT / "scripts"))
DATA = Path(os.environ.get("DRIVING_DATA_ROOT", str(WORK / "dataset/nuscenes")))
PREPARED, ANNOTATIONS, EXTRACT = WORK/"prepared", WORK/"annotations", WORK/"extract"
MODEL = WORK / "models/Cosmos-Reason1-7B"

def run(args):
    args = [str(arg) for arg in args]
    print(shlex.join(args), flush=True)
    subprocess.run(args, cwd=ROOT, check=True)

print("Workspace:", WORK)
print("Verified source files:", len(source_manifest))


## Storage and optional review restore

Colab's `/content` storage is ephemeral: switching runtime can remove it.
Use retained storage via DRIVING_WORK_DIR before bootstrap, or download
the small review backup cell below before switching. Keep this same
notebook to restore that backup; its source fingerprint is checked.
Images and weights are not included in the review backup. The acquisition
cell can reacquire data while preserving the restored split and decisions.


In [ ]:
# After uploading a saved review_handoff.zip to a new runtime, set its path here.
REVIEW_BACKUP_TO_RESTORE = None
if REVIEW_BACKUP_TO_RESTORE is not None:
    from portable_handoff import restore_review
    restored = restore_review(Path(REVIEW_BACKUP_TO_RESTORE), WORK, SOURCE_SHA256)
    print("Restored review files:", len(restored["files_sha256"]))


## 1. Fix the proof-of-concept settings before seeing results

Keep these settings for all four models. A changed protocol should use
a new WORK folder. Do not choose another split after seeing test results.


In [ ]:
SEED, REVIEW_K, EPOCHS, HIDDEN_DIM, TOP_K = 42, 10, 10, 256, 32
LAYER, MAX_PIXELS = 28, 1_600_000
protocol = dict(seed=SEED, review_k=REVIEW_K, epochs=EPOCHS,
                hidden_dim=HIDDEN_DIM, k=TOP_K, layer=LAYER, max_pixels=MAX_PIXELS)
protocol_path = WORK / "requested_protocol.json"
if protocol_path.exists() and json.loads(protocol_path.read_text()) != protocol:
    raise ValueError("Protocol changed; choose a fresh WORK before continuing")
protocol_path.write_text(json.dumps(protocol, indent=2) + "\n")
# CPU data preparation needs no torch or CUDA installation.
run([sys.executable, "-m", "pip", "install", "Pillow", "ipywidgets"])


## 2. Acquire mini, or use an already extracted local copy

The [official tutorial][mini-tutorial]
publishes the mini archive. Its Motional CDN is listed in the
[AWS registry][aws-registry].
Follow the [data terms](https://www.nuscenes.org/terms-of-use).

The downloader records received-byte SHA256, checks transfer completeness
and stages metadata/keyframe camera extraction. It omits sweeps and other
sensors. A local hash is not an independently verified official checksum.
Do not overwrite an existing data folder; preparation validates its files.

[mini-tutorial]: https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuscenes_tutorial.ipynb
[aws-registry]: https://github.com/awslabs/open-data-registry/blob/main/datasets/motional-nuscenes.yaml


In [ ]:
required_tables = ["scene", "sample", "sample_data", "calibrated_sensor", "sensor"]
if all((DATA / "v1.0-mini" / (name + ".json")).is_file()
       for name in required_tables) and (DATA / "samples").is_dir():
    print("Using local data; preparation will validate references and selected images.")
else:
    run([sys.executable, ROOT/"scripts/acquire_nuscenes_mini.py",
         "--archive", WORK/"downloads/v1.0-mini.tgz", "--data-root", DATA,
         "--report", WORK/"acquisition.json"])
if not PREPARED.exists():
    run([sys.executable, ROOT/"scripts/prepare_driving.py", "prepare",
         "--data-root", DATA, "--version", "v1.0-mini", "--frames-per-scene", 8,
         "--seed", SEED, "--rubric-file", ROOT/"latest_grading_criteria.md",
         "--output-dir", PREPARED])
print(json.loads((PREPARED/"prepare_metadata.json").read_text()))


## 3. Review six views on CPU

Enter your reviewer identifier. Each explicit label click saves the CSV
atomically; rerunning this cell reloads it. Use the frozen rubric and keep
uncertain cases uncertain. A model score is not a human label. Preserve
independent review records and adjudicate before importing one final CSV.


In [ ]:
from notebook_review import ReviewSession
review = ReviewSession(PREPARED, WORK/"reviewed.csv")
print(review.progress())
review.widget(DATA)


## 4. Freeze the reviewed labels

Pending/uncertain decisions stay excluded. Each split needs both classes
for this experiment contract. If it does not, keep this PoC as evidence of
the limitation and plan more data; never fabricate labels or re-split by
the test result. Use a new annotations directory for revised labels.


In [ ]:
if not ANNOTATIONS.exists():
    run([sys.executable, ROOT/"scripts/prepare_driving.py", "import-labels",
         "--prepared-dir", PREPARED, "--reviews", WORK/"reviewed.csv",
         "--output-dir", ANNOTATIONS])
label_report = json.loads((ANNOTATIONS/"import_metadata.json").read_text())
current_csv_hash = hashlib.sha256((WORK/"reviewed.csv").read_bytes()).hexdigest()
if list(label_report["review_inputs_sha256"].values()) != [current_csv_hash]:
    raise ValueError("Reviews changed after import; preserve annotations and re-import")
print(label_report["support"])
if not label_report["ready_for_current_training_contract"]:
    raise ValueError("Insufficient reviewed class support; extraction/training blocked")


## Save review progress before changing runtimes

This creates a small backup of prepared records, review CSV, annotations
and the chosen protocol. On Colab it offers a browser download; in other
Jupyter environments retain the printed ZIP using your file browser.
Re-run this cell after recording model/version information to include it.
Preserve feature caches and authorized images separately after extraction.


In [ ]:
from portable_handoff import pack_review
backup_path = WORK / "review_handoff.zip"
pack_review(WORK, backup_path, SOURCE_SHA256)
print("Review backup:", backup_path, "bytes:", backup_path.stat().st_size)
try:
    from google.colab import files as colab_files
except ImportError:
    print("Retain this ZIP before replacing the runtime or workspace.")
else:
    colab_files.download(str(backup_path))


## 5. GPU prerequisites and pinned model

This cell needs an existing CUDA runtime. NVIDIA's
[upstream example](https://github.com/nvidia-cosmos/cosmos-reason1#inference)
calls for a GPU with at least 24 GB. A free Colab GPU is not guaranteed;
no paid upgrade is selected here. The guard below rejects small GPUs,
but passing it does not establish that six full-resolution views fit.

Follow the [Cosmos-Reason1-7B model terms](https://huggingface.co/nvidia/Cosmos-Reason1-7B).
Resolve the official revision once and retain it. A changed pixel budget
or dependency environment requires a fresh cache, not mixed features.


In [ ]:
frozen_review = json.loads((ANNOTATIONS/"import_metadata.json").read_text())
reviewed_hash = hashlib.sha256((WORK/"reviewed.csv").read_bytes()).hexdigest()
if (not frozen_review["ready_for_current_training_contract"]
        or list(frozen_review["review_inputs_sha256"].values()) != [reviewed_hash]):
    raise RuntimeError("Complete and import the reviewed labels before GPU work")
import importlib.metadata, re, shutil

def installed_versions():
    return sorted({f"{d.metadata['Name']}=={d.version}"
                   for d in importlib.metadata.distributions() if d.metadata['Name']})
versions_path = WORK / "gpu_package_versions.txt"
if versions_path.exists():
    if versions_path.read_text() != "\n".join(installed_versions()) + "\n":
        raise RuntimeError("Restore recorded package versions before resuming this cache")
else:
    run([sys.executable, "-m", "pip", "install", "-r", ROOT/"requirements-gpu.txt"])
    versions_path.write_text("\n".join(installed_versions()) + "\n")

# Probe in a fresh process, so pip changes cannot leave stale kernel imports.
probe_code = ("import json,torch; "
              "available=torch.cuda.is_available(); "
              "p=torch.cuda.get_device_properties(0) if available else None; "
              "print(json.dumps(dict(available=available, "
              "name=p.name if p else None, memory=p.total_memory if p else 0, "
              "bf16=torch.cuda.is_bf16_supported() if available else False)))")
probe = subprocess.run([sys.executable, "-c", probe_code],
                       check=True, capture_output=True, text=True)
gpu = json.loads(probe.stdout)
print(gpu)
if not gpu["available"]:
    raise RuntimeError("CUDA is absent. CPU preparation/review remain saved in WORK.")
if gpu["memory"] < 24_000_000_000:
    raise RuntimeError("This guard requires at least 24 billion usable GPU bytes")
DTYPE = "bfloat16" if gpu["bf16"] else "float16"
from huggingface_hub import HfApi, snapshot_download
revision_path = WORK / "model_revision.json"
if revision_path.exists():
    MODEL_REVISION = json.loads(revision_path.read_text())["revision"]
else:
    MODEL_REVISION = HfApi().model_info("nvidia/Cosmos-Reason1-7B").sha
    if not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_REVISION):
        raise ValueError("Model revision is not immutable")
    revision_path.write_text(json.dumps({"model": "nvidia/Cosmos-Reason1-7B",
                                         "revision": MODEL_REVISION}, indent=2))
info = HfApi().model_info("nvidia/Cosmos-Reason1-7B", revision=MODEL_REVISION,
                         files_metadata=True)
model_bytes = sum(item.size or 0 for item in info.siblings)
if (not (MODEL/"config.json").exists()
        and shutil.disk_usage(WORK).free < model_bytes + 4*2**30):
    raise RuntimeError("Insufficient disk for pinned model plus extraction artifacts")
snapshot_download("nvidia/Cosmos-Reason1-7B", revision=MODEL_REVISION, local_dir=MODEL)

run([sys.executable, ROOT/"scripts/driving_preflight.py", "--data-root", DATA,
     "--version", "v1.0-mini", "--model-dir", MODEL,
     "--require", "all", "--output", WORK/"gpu_preflight.json"])


## 6. Measure one frame, then finish the same cache

Use one writer and storage supporting atomic rename/file sync. The cache
retains current/previous generations. Inspect smoke timing and peak VRAM.
An OOM requires a new cache with an explicitly lower fixed pixel budget.
Never train the one-frame smoke cache.


In [ ]:
def extract_command(limit):
    return [sys.executable, ROOT/"scripts/extract.py",
            "--samples-root", DATA/"samples", "--meta-dir", DATA/"v1.0-mini",
            "--model-dir", MODEL, "--model-revision", MODEL_REVISION,
            "--sample-tokens", PREPARED/"selected_samples.json", "--layers", LAYER,
            "--dtype", DTYPE, "--min-pixels", 3136, "--max-pixels", MAX_PIXELS,
            "--save-every", 1, "--max-samples", limit, "--output-dir", EXTRACT]
run(extract_command(1))
smoke_report = WORK/"smoke_extraction_summary.json"
if not smoke_report.exists():
    shutil.copy2(EXTRACT/"extraction_summary.json", smoke_report)
print(json.loads(smoke_report.read_text()))


In [ ]:
if versions_path.read_text() != "\n".join(installed_versions()) + "\n":
    raise RuntimeError("Environment changed since smoke")
run(extract_command(0))
summary = json.loads((EXTRACT/"extraction_summary.json").read_text())
if not summary.get("selection_complete"):
    raise RuntimeError("Extraction is partial; training remains blocked")
run([sys.executable, ROOT/"scripts/run_real_driving.py", "validate",
     "--prepared-dir", PREPARED, "--annotations-dir", ANNOTATIONS,
     "--extract-dir", EXTRACT, "--layer", LAYER,
     "--output", WORK/"handoff_validation.json"])


## 7. Run the controlled CPU comparison

This launches Logistic regression, a small MLP and matched SAE with and
without tail reward. It reuses the imported manifest and checks saved
evidence before training; it never re-splits scenes. Start with one seed.
Preserve exact commands, per-sample predictions and failures. A mini-only
table remains a custom PoC even if the scores look good.


In [ ]:
# Reuse installed torch when continuing on GPU; CPU-only hosts may install it here.
run([sys.executable, "-m", "pip", "install", "torch>=2.7.1", "numpy",
     "scikit-learn", "matplotlib", "Pillow"])
RUNS = WORK / "comparison_runs"
run([sys.executable, ROOT/"scripts/run_real_driving.py", "train",
     "--prepared-dir", PREPARED, "--annotations-dir", ANNOTATIONS,
     "--extract-dir", EXTRACT, "--output-dir", RUNS, "--layer", LAYER,
     "--review-k", REVIEW_K, "--seed", SEED, "--epochs", EPOCHS,
     "--hidden-dim", HIDDEN_DIM, "--k", TOP_K])
print((RUNS/"comparison/main_table.csv").read_text())


## 8. Inspect and retain results

For the browser, run the command below in a terminal that can reach its
localhost server. A notebook output cell does not provide hosted access
to that server. Keep its trained activation cache and authorized images.
Include real FP/FN cases in the final demonstration; random weights and
synthetic examples are not model-effectiveness evidence.

Copy WORK using a transfer method permitted by your environment. Include
hidden `.checkpoints/`, model revision, package versions, frozen prepared
records, review CSV, annotations, extraction and comparison outputs.
The source bundle contains no real data or weights and may be shared
separately. Respect data/model redistribution conditions.


In [ ]:
browser_command = [sys.executable, ROOT/"scripts/neuron_explorer.py", "serve",
                   "--run", RUNS/"sae_abstopk_norm", "--samples-root", DATA/"samples"]
print(shlex.join([str(arg) for arg in browser_command]))
print("Keep the complete work directory:", WORK)
